# Waze vs the incident register · checkpoint 1

Catch-up notebook: one way to get here, with its output. Done so far: the feed read, one row per alert.

Run all the cells (Runtime › Run all), then continue the exercise from the end of this notebook with your own code.

In [1]:
# The tools for today: pandas for tables, geopandas for maps and distances, matplotlib for plots.
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Where the data files are: next to the notebook if there is a data folder,
# otherwise read straight from the course repository on GitHub (this is the case in Colab).
DATA = "https://raw.githubusercontent.com/daniel-verdugo/upm-data-integration/main/3.1-data-integration/session14/data/"
if os.path.isdir("data"):
    DATA = "data/"
pd.set_option("display.width", 140)
plt.rcParams["figure.dpi"] = 150   # sharper plots
pd.set_option("display.max_columns", 30)

## 1 · The feed: what is one row?

One line of `waze_feed.jsonl` is one poll of the partner feed, every 5 minutes, exactly as Waze
returns it. Field meanings: the official *Waze Data Feed specifications*
(https://support.google.com/waze/partners/answer/13458165).

In [2]:
# Read the Waze feed. The file has one JSON object per line, and each line is one poll.
# shape tells me how many polls (rows) and fields (columns) I have.
polls = pd.read_json(DATA + "waze_feed.jsonl", lines=True)
polls.shape

(12096, 6)

In [3]:
# A first look: each poll has its time window and, inside, a list of alerts and a list of jams.
polls.head()

,endTimeMillis,startTimeMillis,startTime,endTime,alerts,jams
0,1769986835790,1769986775790,2026-02-01 22:59:35:000,2026-02-01 23:00:35:000,NaN,NaN
1,1769987131479,1769987071479,2026-02-01 23:04:31:000,2026-02-01 23:05:31:000,NaN,NaN
2,1769987435609,1769987375609,2026-02-01 23:09:35:000,2026-02-01 23:10:35:000,NaN,NaN
3,1769987730105,1769987670105,2026-02-01 23:14:30:000,2026-02-01 23:15:30:000,NaN,NaN
4,1769988038087,1769987978087,2026-02-01 23:19:38:000,2026-02-01 23:20:38:000,NaN,NaN


Each poll carries a list of alerts. `explode` gives one row per alert in a poll:

In [4]:
# I want one row per alert. explode splits each list of alerts into separate rows.
# Polls without alerts leave an empty row, so I drop those.
alert_rows = polls[["endTimeMillis", "alerts"]].explode("alerts")
alert_rows = alert_rows.dropna(subset=["alerts"])
alert_rows.shape

(7618, 2)

In [5]:
# Each alert is still a dictionary. json_normalize turns its fields into columns
# (location becomes location.x and location.y). I also keep the poll time, to know when it was seen.
alerts = pd.json_normalize(alert_rows["alerts"].tolist())
alerts["poll_ms"] = alert_rows["endTimeMillis"].values
alerts.head()

,country,reportRating,reportByMunicipalityUser,reliability,type,uuid,roadType,magvar,subtype,pubMillis,confidence,nThumbsUp,street,location.x,location.y,poll_ms
0,SP,3,false,8,HAZARD,024e0f17-8e37-48fe-97f3-bf85b5299431,3,73,HAZARD_ON_ROAD_CAR_STOPPED,1769989182613,0,0,AP-34,-4.902474,41.646961,1769989535119
1,SP,3,false,8,HAZARD,024e0f17-8e37-48fe-97f3-bf85b5299431,3,73,HAZARD_ON_ROAD_CAR_STOPPED,1769989182613,0,0,AP-34,-4.902474,41.646961,1769989838479
2,SP,3,false,8,HAZARD,024e0f17-8e37-48fe-97f3-bf85b5299431,3,73,HAZARD_ON_ROAD_CAR_STOPPED,1769989182613,0,0,AP-34,-4.902474,41.646961,1769990130332
3,SP,3,false,8,HAZARD,024e0f17-8e37-48fe-97f3-bf85b5299431,3,73,HAZARD_ON_ROAD_CAR_STOPPED,1769989182613,0,0,AP-34,-4.902474,41.646961,1769990435299
4,SP,3,false,8,HAZARD,024e0f17-8e37-48fe-97f3-bf85b5299431,3,73,HAZARD_ON_ROAD_CAR_STOPPED,1769989182613,0,0,AP-34,-4.902474,41.646961,1769990735345


In [6]:
# How many different alerts are there really? Fewer than rows: the same uuid repeats.
alerts["uuid"].nunique()

1129

One row is **one alert in one poll**. An alert stays in the feed while it is active, so the same
`uuid` appears in many polls, and its `reliability` and `confidence` change along the way.

In [7]:
# In how many polls does each alert appear? This confirms the grain: one alert in one poll.
alerts.groupby("uuid").size().describe()

count    1129.000000
mean        6.747564
std         4.421403
min         1.000000
25%         3.000000
50%         6.000000
75%        10.000000
max        24.000000
dtype: float64

## 2 · From polls to alerts (one row per `uuid`)

In [8]:
# Now one row per alert. I group the rows of each uuid and keep the first value of the fields
# that do not change, the first and last poll it was seen in, and the highest reliability
# and confidence it reached.
ev = alerts.sort_values("poll_ms").groupby("uuid").agg(
    type=("type", "first"),
    subtype=("subtype", "first"),
    street=("street", "first"),
    pubMillis=("pubMillis", "first"),
    first_poll=("poll_ms", "min"),
    last_poll=("poll_ms", "max"),
    reliability=("reliability", "max"),
    confidence=("confidence", "max"),
    x=("location.x", "first"),
    y=("location.y", "first"),
    magvar=("magvar", "first"),
)
len(ev)

1129

In [9]:
# What kinds of alerts are there? Many of them are probably not incidents the operator logs.
ev.groupby(["type", "subtype"]).size().sort_values(ascending=False)

type      subtype                       
HAZARD    HAZARD_ON_SHOULDER_CAR_STOPPED    430
          HAZARD_ON_ROAD_LANE_CLOSED        110
          HAZARD_ON_ROAD_CAR_STOPPED         94
          HAZARD_ON_ROAD_CONSTRUCTION        88
JAM       JAM_HEAVY_TRAFFIC                  71
HAZARD    HAZARD_ON_ROAD_OBJECT              67
ACCIDENT                                     54
JAM       JAM_STAND_STILL_TRAFFIC            53
                                             48
HAZARD    HAZARD_ON_ROAD                     32
          HAZARD_WEATHER_FOG                 28
          HAZARD_WEATHER                     18
          HAZARD_ON_ROAD_POT_HOLE            14
ACCIDENT  ACCIDENT_MAJOR                     10
HAZARD    HAZARD_ON_SHOULDER_ANIMALS          9
ACCIDENT  ACCIDENT_MINOR                      3
dtype: int64

**End of checkpoint 1.** From here on, the work is yours.